## 1. Import Libraries

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding

C:\Users\DELL\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


 ## 2. Load Dataset

In [2]:
with open("shakespeare.txt", "r", encoding='utf-8') as file:
    data = file.read().lower()

print(data[:500])

first citizen:
before we proceed any further, hear me speak.

all:
speak, speak.

first citizen:
you are all resolved rather to die than to famish?

all:
resolved. resolved.

first citizen:
first, you know caius marcius is chief enemy to the people.

all:
we know't, we know't.

first citizen:
let us kill him, and we'll have corn at our own price.
is't a verdict?

all:
no more talking on't; let it be done: away, away!

second citizen:
one word, good citizens.

first citizen:
we are accounted poor


## 3. Tokenization

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)

Total words: 12633


## 4. Create Input Sequences

In [4]:
input_sequences = []

for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

print("Total sequences:", len(input_sequences))

Total sequences: 171312


## 5. Padding

In [5]:
max_seq_len = max([len(x) for x in input_sequences])
print("Max sequence length:", max_seq_len)

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')
)

print(input_sequences.shape)

Max sequence length: 16
(171312, 16)


## 6. Split Data

In [12]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

# y = tf.keras.utils.to_categorical(y, num_classes=total_words)
# print("y after one-hot:", y.shape)

X shape: (171312, 15)
y shape: (171312,)


## 7. Build Model

In [13]:
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [14]:
model = Sequential([
    Embedding(input_dim=total_words, output_dim=100, input_length=max_seq_len-1),
    LSTM(150),
    Dense(total_words, activation='softmax')
])

In [15]:
model.build(input_shape=(None, max_seq_len-1))
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ (None, 15, 100)             │       1,263,300 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ (None, 150)                 │         150,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 12633)               │       1,907,583 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,321,483 (12.67 MB)

 Trainable params: 3,321,483 (12.67 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

## 8. Train Model

In [18]:
history = model.fit(X, y, epochs=5, verbose=1)

Epoch 1/5
5354/5354 ━━━━━━━━━━━━━━━━━━━━ 286s 53ms/step - accuracy: 0.0559 - loss: 6.6819
Epoch 2/5
5354/5354 ━━━━━━━━━━━━━━━━━━━━ 230s 43ms/step - accuracy: 0.0946 - loss: 6.0097
Epoch 3/5
5354/5354 ━━━━━━━━━━━━━━━━━━━━ 226s 42ms/step - accuracy: 0.1101 - loss: 5.6559
Epoch 4/5
5354/5354 ━━━━━━━━━━━━━━━━━━━━ 219s 41ms/step - accuracy: 0.1233 - loss: 5.3397
Epoch 5/5
5354/5354 ━━━━━━━━━━━━━━━━━━━━ 223s 42ms/step - accuracy: 0.1370 - loss: 5.0345


## 9. Save Model

In [19]:
model.save("text_gen_model.h5")

## 10. Save Tokenizer

In [20]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

## 11. Text Generation Function

In [21]:
def generate_text(seed_text, next_words=10):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
        
        predicted = np.argmax(model.predict(token_list), axis=-1)
        
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
                
    return seed_text

## 12. Test Output

In [22]:
print(generate_text("love is", 10))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
love is the matter to the people and the people will not
